# 22-35 · Приёмочная проверка итогового проекта

Практика к разделу [«Итоговый проект: список задач на Flask и SQLite»](../../site/chapters/glava-22/22-35-itogovyj-proekt.html). Использует настоящий `projects/flask/todo-app/app.py`.

## Цель

Пройти по итоговому проекту от начала до конца: добавить задачу, отметить её выполненной, проверить API и persistence — на реальном приложении, во временной базе данных.

## Рабочий пример

In [ ]:
import sys
import tempfile
import os

sys.path.insert(0, "../../projects/flask/todo-app")

os.environ["TODO_APP_DB"] = os.path.join(tempfile.mkdtemp(), "itogovyj_test.db")

import app as todo_app

with todo_app.app.app_context():
    todo_app.init_db()

client = todo_app.app.test_client()

otvet_pustoj = client.get("/")
print("Пустой список:", "Задач пока нет" in otvet_pustoj.get_data(as_text=True))

client.post("/dobavit", data={"zadacha": "Проверить итоговый проект"}, follow_redirects=True)
otvet_api = client.get("/api/tasks")
zadachi = otvet_api.get_json()
print("После добавления:", zadachi)

## Проверка результата

In [ ]:
assert "Задач пока нет" in otvet_pustoj.get_data(as_text=True)
assert len(zadachi) == 1
assert zadachi[0]["title"] == "Проверить итоговый проект"
assert zadachi[0]["done"] is False
print("Верно: задача добавлена и видна через HTML-страницу и через JSON API.")

## Эксперимент — отметка выполнения и persistence

In [ ]:
task_id = zadachi[0]["id"]
client.post(f"/vypolnit/{task_id}", follow_redirects=True)

otvet_posle = client.get("/api/tasks").get_json()
assert otvet_posle[0]["done"] is True
print("Верно: задача отмечена выполненной.")

# "перезапуск": создаём новый объект приложения на том же файле базы данных
del sys.modules["app"]
import app as todo_app2

client2 = todo_app2.app.test_client()
otvet_posle_perezapuska = client2.get("/api/tasks").get_json()

assert len(otvet_posle_perezapuska) == 1
assert otvet_posle_perezapuska[0]["done"] is True
print("Верно: данные пережили создание нового объекта приложения — то есть ведут себя так же, как после перезапуска процесса.")